In [ ]:
!unzip "pakiet(1).zip" -d lab5

Archive:  pakiet(1).zip
   creating: lab5/pakiet/
  inflating: lab5/__MACOSX/._pakiet  
  inflating: lab5/pakiet/train.pkl   
  inflating: lab5/__MACOSX/pakiet/._train.pkl  
  inflating: lab5/pakiet/.DS_Store   
  inflating: lab5/__MACOSX/pakiet/._.DS_Store  
  inflating: lab5/pakiet/test_no_target.pkl  
  inflating: lab5/__MACOSX/pakiet/._test_no_target.pkl  
  inflating: lab5/pakiet/treść_zadania.txt  
  inflating: lab5/__MACOSX/pakiet/._treść_zadania.txt  


In [ ]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch
import pickle
import os
import random


PAD_VALUE = 0.0

class MusicAcordsDataset(Dataset):
    def __init__(self, data, has_targets=True):
        super().__init__()
        self.sequences = []
        self.classes = []
        self.has_targets = has_targets

        for sample in data:
            if self.has_targets:
                seq = sample[0]
                self.classes.append(torch.tensor(sample[1], dtype=torch.long))
            else:
                seq = sample

            self.sequences.append(torch.tensor(seq, dtype=torch.float32).unsqueeze(1)) # [seq_length, 1]

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, indx):
        if self.has_targets:
            return self.sequences[indx], self.classes[indx]
        return self.sequences[indx], None # For test dataset

def pad_collate(batch):
    xx = [item[0] for item in batch]
    yy = [item[1] for item in batch if item[1] is not None]

    x_lens = torch.tensor([len(x) for x in xx], dtype=torch.long)
    xx_pad = pad_sequence(xx, batch_first=True, padding_value=PAD_VALUE)

    if len(yy) > 0:
        yy_stack = torch.stack(yy)
        return xx_pad, yy_stack, x_lens

    return xx_pad, None, x_lens # for test dataset


with open(os.path.join("lab5", "pakiet", "test_no_target.pkl"), 'rb') as f:
    test_data = pickle.load(f)

with open(os.path.join("lab5", "pakiet", "train.pkl"), 'rb') as f:
    data = pickle.load(f)


random.seed(42)
random.shuffle(data)

train_size = int(0.7 * len(data))
train_data = data[:train_size]
val_data = data[train_size:]


train_set = MusicAcordsDataset(train_data)
val_set = MusicAcordsDataset(val_data)
test_set = MusicAcordsDataset(test_data, has_targets=False)

train_loader = DataLoader(train_set, batch_size=32, shuffle=True, collate_fn=pad_collate)
val_loader = DataLoader(val_set, batch_size=132, shuffle=False, drop_last=False, collate_fn=pad_collate)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False, drop_last=False, collate_fn=pad_collate)

In [ ]:
[len(train_data[i][0]) for i in range(len(train_data))]

In [ ]:
print(torch.tensor([len(train_data[i][0]) for i in range(len(train_data))]).float().mean())

tensor(431.3763)


In [ ]:
max_v = float("-inf")
for seq in val_data:
    # Zakładając, że seq to tensor PyTorcha:
    current_max = seq[0].max().item()
    if current_max > max_v:
        max_v = current_max

print(max_v)

191.0


In [ ]:
print(train_data[0][0])

[112. 112.   0.  88.  88.  67.  67. 180. 180. 114.  67.  67.   0.   0.
   0.  67.  88.  33.  33.  68. 159.  82.  41. 172.  28.   0.  47. 180.
  47.   0.   0.  67.  73.  41.  73.  20.  36.   2.   2.  36.  88.   0.
  67.  67.  67.  64.  67.   0.  82.  12.  73. 112. 190. 148. 172. 180.
  12.  92. 100.   0.   0.   0.  67.   0.  82.  12.  73. 112. 190. 148.
 172. 180.  12.  92. 100.   0.   0.   0.  67.  67. 180.  20.  33.  33.
  33.  33.  80.  80.  84. 100.  71. 180. 180. 180. 180. 180.  12. 180.
  41. 145. 145. 145.  68.  68.  68.  68. 154. 148. 148. 148. 148. 148.
   8.  12.  73. 112. 190. 148. 190. 112.  12. 124. 132. 145. 148.  32.
 114. 114. 148.  12.  73. 112. 190. 148. 190. 112.  12. 124. 132. 145.
 148.  32. 114. 114. 148.  12.  73. 112. 190. 148. 190. 112. 120. 153.
 156. 132.   3.  67.   0.   0. 154. 120.  18.  32.  32.  32.  32.  32.
  20.   0.  52.  79.  20. 180.  73.  20.  68.  52.  45. 190. 185. 132.
 100. 185. 124.   8.   8. 185. 100. 100. 100. 100.  88.  67.  67. 145.
 190. 

In [ ]:
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import torch.nn as nn
import torch

# Hiperparameters for experiments
LSTM_HIDDEN_SIZE = 128
CLASSIFIER_HIDDEN_SIZE = 128
LSTM_NUM_LAYERS = 2
MEAN_POOLING = True
BIDIRECTIONAL = True


class LSTM_Seq_Regressor(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, bidirectional):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(
            input_size = input_size,
            hidden_size = hidden_size,
            num_layers = num_layers,
            batch_first = True,
            bidirectional = bidirectional
        )

    def forward(self, x):
        all_outputs, (h_n, c_n) = self.lstm(x)
        return all_outputs, (h_n, c_n)


class MusicAuthorClassifier(nn.Module):
    def __init__(self, seq_model: LSTM_Seq_Regressor, num_classes, hidden_size, use_mean_pooling, bidirectional):
        super().__init__()
        self.use_mean_pooling = use_mean_pooling
        self.seq_model = seq_model
        self.bidirectional = bidirectional

        input_size = seq_model.hidden_size * 2 if bidirectional else seq_model.hidden_size

        self.classification_head = nn.Sequential(
            nn.BatchNorm1d(input_size),
            nn.Dropout(p=0.3),
            nn.Linear(input_size, hidden_size),
            nn.GELU(),
            nn.BatchNorm1d(hidden_size),
            nn.Dropout(p=0.3),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x, x_len):
        x_packed = pack_padded_sequence(x, x_len.cpu(), batch_first=True, enforce_sorted=False)

        all_outputs, (h_n, c_n) = self.seq_model(x_packed)

        if self.use_mean_pooling:
            unpacked_out, _ = pad_packed_sequence(all_outputs, batch_first=True) # [batch, seq, hidden_size]

            sum_hidden = torch.sum(unpacked_out, dim=1)
            pooled_hidden = sum_hidden / x_len.unsqueeze(1).float().to(sum_hidden.device)

            out = self.classification_head(pooled_hidden)

        elif self.bidirectional:
            forward_hidden = h_n[-2]
            backward_hidden = h_n[-1]
            last_hidden = torch.cat((forward_hidden, backward_hidden), dim=1)   # [batch, seq, 2*hidden_size]
            out = self.classification_head(last_hidden)

        else:
            last_hidden = h_n[-1] # [batch, seq, hidden_size]
            out = self.classification_head(last_hidden)

        return out


device = "cuda" if torch.cuda.is_available() else "cpu"

lstm_base = LSTM_Seq_Regressor(input_size=1, hidden_size=LSTM_HIDDEN_SIZE, num_layers=LSTM_NUM_LAYERS, bidirectional=BIDIRECTIONAL)

model = MusicAuthorClassifier(seq_model=lstm_base, num_classes=5, hidden_size=CLASSIFIER_HIDDEN_SIZE, use_mean_pooling=MEAN_POOLING, bidirectional=BIDIRECTIONAL).to(device)

In [ ]:
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import torch.nn as nn
import torch

EPOCHS = 100

def train():
    loss_func = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    for epoch in range(EPOCHS):
        model.train()

        train_epoch_loss = 0.0

        for x, targets, x_len in train_loader:
            x, targets = x.to(device), targets.to(device)

            optimizer.zero_grad()

            preds = model(x, x_len)

            loss = loss_func(preds, targets)
            loss.backward()
            optimizer.step()

            train_epoch_loss += loss.item()

        if epoch % 10 == 0:
            model.eval()

            all_preds = []
            all_targets = []
            val_loss = 0.0

            for x, targets, x_len in val_loader:
                x, targets = x.to(device), targets.to(device)

                with torch.no_grad():
                    logits = model(x, x_len)

                    predicted_classes = torch.argmax(logits, dim=1)

                    all_preds.append(predicted_classes)
                    all_targets.append(targets)
                    loss = loss_func(logits, targets)
                    val_loss += loss.item()

            all_preds = torch.cat(all_preds)
            all_targets = torch.cat(all_targets)

            val_acc = torch.sum(all_preds == all_targets).float() / len(all_preds)

            print(f"Epoch: {epoch}, train loss: {train_epoch_loss/len(train_loader):.3f}, val_loss: {val_loss/len(val_loader)}, val_acc: {val_acc}")

## Normal

In [ ]:
train()

Epoch: 0, train loss: 1.658, val_loss: 1.2469138247626168, val_acc: 0.5646258592605591
Epoch: 10, train loss: 1.058, val_loss: 0.982551885502679, val_acc: 0.6485260725021362
Epoch: 20, train loss: 0.868, val_loss: 0.8590713015624455, val_acc: 0.6712018251419067
Epoch: 30, train loss: 0.734, val_loss: 0.7000790578978402, val_acc: 0.7709750533103943
Epoch: 40, train loss: 0.673, val_loss: 0.7573697524411338, val_acc: 0.7188208699226379
Epoch: 50, train loss: 0.506, val_loss: 0.7824391296931675, val_acc: 0.7664399147033691
Epoch: 60, train loss: 0.435, val_loss: 0.8056732054267611, val_acc: 0.7675737142562866
Epoch: 70, train loss: 0.373, val_loss: 0.6608290672302246, val_acc: 0.7879818677902222
Epoch: 80, train loss: 0.301, val_loss: 0.7316091273512159, val_acc: 0.7925170063972473
Epoch: 90, train loss: 0.292, val_loss: 0.8172411407743182, val_acc: 0.7811791300773621


In [ ]:
import torch
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"
model.eval()
all_preds = []

with torch.no_grad():
    for x, _, x_len in test_loader:
        x = x.to(device)
        logits = model(x, x_len)
        predicted_classes = torch.argmax(logits, dim=1)
        all_preds.extend(predicted_classes.cpu().numpy())

df = pd.DataFrame(data=all_preds)
df.to_csv("test_predictions.csv", index=False)


In [ ]:
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import torch.nn as nn
import torch
import os

EPOCHS = 100


def train():
    loss_func = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=4, min_lr=1e-6
    )

    device = "cuda" if torch.cuda.is_available() else "cpu"

    best_val_loss = float('inf')
    save_path = 'best_model.pth'

    for epoch in range(EPOCHS):
        # TRAINING
        model.train()
        train_epoch_loss = 0.0

        for x, targets, x_len in train_loader:
            x, targets = x.to(device), targets.to(device)

            optimizer.zero_grad()

            preds = model(x, x_len)

            loss = loss_func(preds, targets)
            loss.backward()
            optimizer.step()

            train_epoch_loss += loss.item()

        # VALIDATION
        model.eval()

        all_preds = []
        all_targets = []
        val_loss_total = 0.0

        for x, targets, x_len in val_loader:
            x, targets = x.to(device), targets.to(device)

            with torch.no_grad():
                logits = model(x, x_len)

                predicted_classes = torch.argmax(logits, dim=1)

                all_preds.append(predicted_classes)
                all_targets.append(targets)
                loss = loss_func(logits, targets)
                val_loss_total += loss.item()

        all_preds = torch.cat(all_preds)
        all_targets = torch.cat(all_targets)

        avg_train_loss = train_epoch_loss / len(train_loader)
        avg_val_loss = val_loss_total / len(val_loader)
        val_acc = torch.sum(all_preds == all_targets).float() / len(all_preds)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), save_path)
            saved_info = f"| The best model saved"
        else:
            saved_info = ""

        scheduler.step(avg_val_loss)
        current_lr = optimizer.param_groups[0]['lr']

        print(f"Epoch: {epoch:3d} | LR: {current_lr:.6f} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}{saved_info}")

## Bidirectional

In [ ]:
train()

Epoch:   0 | Train Loss: 1.4179 | Val Loss: 1.2359 | Val Acc: 0.5748 ---> NAJLEPSZY MODEL ZAPISANY!
Epoch:   1 | Train Loss: 1.1741 | Val Loss: 1.3393 | Val Acc: 0.4467
Epoch:   2 | Train Loss: 1.0737 | Val Loss: 0.9347 | Val Acc: 0.6723 ---> NAJLEPSZY MODEL ZAPISANY!
Epoch:   3 | Train Loss: 0.9920 | Val Loss: 1.0269 | Val Acc: 0.6406
Epoch:   4 | Train Loss: 0.9484 | Val Loss: 0.9109 | Val Acc: 0.7222 ---> NAJLEPSZY MODEL ZAPISANY!
Epoch:   5 | Train Loss: 0.9497 | Val Loss: 0.8487 | Val Acc: 0.7007 ---> NAJLEPSZY MODEL ZAPISANY!
Epoch:   6 | Train Loss: 0.9214 | Val Loss: 1.0408 | Val Acc: 0.6156
Epoch:   7 | Train Loss: 0.8708 | Val Loss: 0.8569 | Val Acc: 0.7018
Epoch:   8 | Train Loss: 0.8627 | Val Loss: 0.7854 | Val Acc: 0.7177 ---> NAJLEPSZY MODEL ZAPISANY!
Epoch:   9 | Train Loss: 0.8562 | Val Loss: 1.0042 | Val Acc: 0.6678
Epoch:  10 | Train Loss: 0.8237 | Val Loss: 0.8234 | Val Acc: 0.7154
Epoch:  11 | Train Loss: 0.8261 | Val Loss: 0.7730 | Val Acc: 0.7302 ---> NAJLEPSZY MO

## Birectional & mean pooling

In [ ]:
train()

Epoch:   0 | LR: 0.001000 | Train Loss: 1.4826 | Val Loss: 1.3368 | Val Acc: 0.5680| The best model saved
Epoch:   1 | LR: 0.001000 | Train Loss: 1.2402 | Val Loss: 1.2787 | Val Acc: 0.6134| The best model saved
Epoch:   2 | LR: 0.001000 | Train Loss: 1.1896 | Val Loss: 1.1190 | Val Acc: 0.6848| The best model saved
Epoch:   3 | LR: 0.001000 | Train Loss: 1.1609 | Val Loss: 0.9470 | Val Acc: 0.6599| The best model saved
Epoch:   4 | LR: 0.001000 | Train Loss: 1.1192 | Val Loss: 0.9087 | Val Acc: 0.6576| The best model saved
Epoch:   5 | LR: 0.001000 | Train Loss: 0.9992 | Val Loss: 0.8518 | Val Acc: 0.6927| The best model saved
Epoch:   6 | LR: 0.001000 | Train Loss: 1.0084 | Val Loss: 0.8226 | Val Acc: 0.7109| The best model saved
Epoch:   7 | LR: 0.001000 | Train Loss: 0.9293 | Val Loss: 0.8110 | Val Acc: 0.7347| The best model saved
Epoch:   8 | LR: 0.001000 | Train Loss: 0.9565 | Val Loss: 0.9407 | Val Acc: 0.6497
Epoch:   9 | LR: 0.001000 | Train Loss: 0.9065 | Val Loss: 0.9203 | 

## Mean pooling outputs

In [ ]:
train()